[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C72_MultiView_Geometry_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与一条完整的投影链

这个 notebook 做三件事：

1. **造一个可控的合成路口** —— 一个地平面、三块装在不同高度的标志、一台前视相机。
   每个点的三维真值我都知道，所以每个误差都能被断言。
2. **把投影链正反跑通**，并且把「反投影用了什么假设」做成一个**必填参数**。
3. **推出并穷举验证本课的中心公式** $d_{\text{read}} = d\cdot H/(H-z)$，
   然后把「1° 外参误差」换算成「多少个像素的定位误差」。

> 心智模型：**正向投影唯一确定，反向投影永远缺一个自由度。
> 你每次反投影，都在用一个假设补那个自由度——而这门课讲的是那些假设什么时候崩。**

## 0 · 环境与坐标系约定

坐标系约定是几何代码里第一大 bug 来源，所以先钉死，全课不变：

| 坐标系 | 约定 | 说明 |
|---|---|---|
| 车体 | **X 前 · Y 左 · Z 上** | ROS REP-103，自动驾驶栈的事实标准 |
| 相机 | **x 右 · y 下 · z 光轴向前** | 计算机视觉标准（OpenCV 同款） |
| 图像 | `u` 向右 · `v` 向下 · 原点左上 | 像素 |

**注意车体的 Y 是「左」而相机的 x 是「右」——这两个约定天生反向**，
所以旋转矩阵第一列是 `(0,-1,0)`。忘掉这个符号会得到一个左右镜像的世界，
而它在正前方的目标上**看不出来**。

In [ ]:
import numpy as np
import itertools

print('numpy', np.__version__)
print('本课全程 CPU / 断网 / 不训练任何网络 / 不依赖 OpenCV')

# ── 相机与场景的物理参数（全课通用）──
H  = 1.5       # 相机离地高度 (m)
F  = 1200.0    # 焦距 (px)
W, HGT = 1920, 1080
CX, CY = W / 2, HGT / 2

K = np.array([[F, 0, CX],
              [0, F, CY],
              [0, 0,  1]])

def R_vc(pitch_deg=0.0):
    """车体(X前,Y左,Z上) ← 相机(x右,y下,z前) 的旋转矩阵。pitch>0 表示低头。

    列 = 相机三个轴在车体坐标里的方向：
      x_c(图像右)  = (0, -1, 0)            ← 图像右 = 车体右 = -Y
      y_c(图像下)  = (-sinθ, 0, -cosθ)
      z_c(光轴)    = ( cosθ, 0, -sinθ)
    """
    t = np.deg2rad(pitch_deg)
    return np.array([[0., -np.sin(t),  np.cos(t)],
                     [-1.,        0.,        0.],
                     [0., -np.cos(t), -np.sin(t)]])

CAM_T = np.array([0., 0., H])   # 相机光心在车体坐标里的位置

# 右手性自检：三个轴必须构成右手系，否则世界会被左右镜像
for p in [0.0, 1.0, -3.0, 12.0]:
    assert abs(np.linalg.det(R_vc(p)) - 1.0) < 1e-12, 'R 必须是旋转矩阵(det=+1)'
    assert np.allclose(R_vc(p) @ R_vc(p).T, np.eye(3), atol=1e-12)
print('✅ 旋转矩阵在各 pitch 下都是正交且 det=+1')

## 1 · 一个合成路口

地面上一排车道标记（`z=0`），加三块**装在不同高度**的标志。
高度取值不是随便挑的：**限速牌牌面中心通常离地 2.0–2.5 m，
而乘用车前视相机在 1.2–1.5 m** —— 这个高度关系是本课中心结论的全部来源。

In [ ]:
SCENE = {
    # 名字: (X前, Y左, Z上)  —— 单位 m
    'lane_10m':   (10.0,  1.75, 0.00),   # 车道线上的一个标记点
    'lane_30m':   (30.0,  1.75, 0.00),
    'lane_60m':   (60.0,  1.75, 0.00),
    'arrow_20m':  (20.0,  0.00, 0.00),   # 路面箭头
    'sign_low':   (30.0, -3.20, 1.00),   # 一块很低的牌（施工牌）
    'sign_speed': (30.0, -3.20, 2.20),   # 限速牌：**高于相机**
    'sign_gantry':(30.0,  0.00, 5.50),   # 龙门架上的悬挂标志
}

def project(P_v, pitch=0.0):
    """车体坐标点 -> 像素。返回 None 表示在相机后方。"""
    P_c = R_vc(pitch).T @ (np.asarray(P_v, float) - CAM_T)
    if P_c[2] <= 1e-9:
        return None
    return np.array([CX + F * P_c[0] / P_c[2],
                     CY + F * P_c[1] / P_c[2]])

print(f"{'目标':12s} {'真值 (X,Y,Z)':>22s} {'像素 (u,v)':>18s}  位置")
for name, P in SCENE.items():
    uv = project(P)
    where = '地平线以上' if uv[1] < CY else '地平线以下'
    print(f'{name:12s} {str(tuple(P)):>22s}  ({uv[0]:7.1f},{uv[1]:7.1f})  {where}')

# 地面点必然成像在地平线（v=CY）**以下**；高于相机的点必然在**以上**
for name, P in SCENE.items():
    v = project(P)[1]
    if P[2] < H:  assert v > CY, f'{name} 低于相机 -> 应在地平线以下'
    if P[2] > H:  assert v < CY, f'{name} 高于相机 -> 应在地平线以上'
print('\n✅ 地平线把场景分成两半：低于相机高度的在下，高于的在上')

## 2 · 反投影：把「假设」做成必填参数

一个像素对应**一条射线**，不是一个点。要落到三维必须补一个自由度，
而补法只有四种。**本课所有反投影函数都强制传 `assumption=`** ——
把假设变成必填参数，是让它不被忘记的最便宜的办法。

In [ ]:
def backproject(uv, assumption, pitch=0.0, z=None, size_px=None, size_m=None):
    """像素 -> 车体坐标。assumption 必填，取值：

      'ground'        : 假设目标在 z=0 的地面上
      'known_height'  : 已知目标离地高度 z（需要 z=）
      'known_size'    : 已知目标物理尺寸与像素尺寸（需要 size_px=, size_m=）
    """
    d_c = np.array([(uv[0] - CX) / F, (uv[1] - CY) / F, 1.0])
    d_v = R_vc(pitch) @ d_c                      # 射线方向（车体坐标）

    if assumption in ('ground', 'known_height'):
        plane_z = 0.0 if assumption == 'ground' else float(z)
        # 求射线与平面 Z=plane_z 的交点： CAM_T[2] + s*d_v[2] = plane_z
        denom = d_v[2]
        if abs(denom) < 1e-12 or (H - plane_z) / denom > 0:
            return None                          # 平行或朝错方向 -> 无交点
        s = (plane_z - H) / denom
        return CAM_T + s * d_v

    if assumption == 'known_size':
        # 相似三角形：真实边长 / 像素边长 = 深度 / 焦距
        depth = F * float(size_m) / float(size_px)
        return CAM_T + (depth / d_v[0]) * d_v if abs(d_v[0]) > 1e-12 else None

    raise ValueError(f'未知假设 {assumption!r} —— 假设必须显式写出来')

# 闭环：地面点用 ground 假设，必须精确回到原处
for name in ['lane_10m', 'lane_30m', 'lane_60m', 'arrow_20m']:
    P = np.array(SCENE[name]); back = backproject(project(P), 'ground')
    assert np.allclose(back, P, atol=1e-9), (name, back, P)
print('✅ 地面点闭环精确（误差 < 1e-9 m）—— 因为假设与真相一致')

# 而高于相机的标志，ground 假设直接**无解**
for name in ['sign_speed', 'sign_gantry']:
    assert backproject(project(SCENE[name]), 'ground') is None
print('✅ 高于相机的标志：ground 假设返回 None，不是「误差大」而是**没有交点**')

# 换成正确的假设就精确了
P = np.array(SCENE['sign_speed'])
back = backproject(project(P), 'known_height', z=P[2])
assert np.allclose(back, P, atol=1e-9)
print('✅ 同一个像素 + 正确的高度假设 -> 精确复原', np.round(back, 6))

## 3 · 中心公式：地面假设被破坏时读出什么

设相机高 $H$、目标真实纵向距离 $d$、目标离地 $z$。
目标点相对相机的俯角是 $\theta=\arctan\frac{H-z}{d}$；
地面假设沿同一条射线求 $z=0$ 的交点，得到

$$d_{\text{read}} = \frac{H}{\tan\theta} = d\cdot\frac{H}{H-z}$$

下面用逐点数值穷举来验证它。

In [ ]:
def ipm_read_formula(d, z, h=H):
    """地面假设读出的纵向距离。z>=h 时无解，返回 inf。"""
    if z >= h:
        return np.inf
    return d * h / (h - z)

max_dev = 0.0
rows = []
for d in [8, 10, 20, 30, 50, 80, 120]:
    for z in [0.0, 0.3, 0.8, 1.0, 1.2, 1.4, 1.49]:
        bp = backproject(project((d, 0.0, z)), 'ground')
        num = np.inf if bp is None else bp[0]
        pred = ipm_read_formula(d, z)
        if np.isfinite(num) and np.isfinite(pred):
            max_dev = max(max_dev, abs(num - pred) / pred)   # **相对**偏差
        rows.append((d, z, num, pred))

print(f'穷举 {len(rows)} 组 (d,z)，公式与数值的最大**相对**偏差 = {max_dev:.3e}')
assert max_dev < 1e-12, '公式必须与逐点数值一致'
# 注意这里必须用相对偏差：z=1.49 时 d_read 到 18000 m，
# 浮点绝对误差可达 4e-9，而相对误差只有 2e-13——**绝对容差在这里是错的量纲**

print(f"\n{'d 真':>6s} {'z':>5s} {'数值读出':>11s} {'公式':>11s}")
for d, z, num, pred in rows:
    if d in (10, 50) and z in (0.0, 0.3, 1.0, 1.4):
        print(f'{d:6d} {z:5.1f} {num:11.4f} {pred:11.4f}')

# z >= H 无解
for z in [1.5, 2.2, 5.5]:
    assert backproject(project((30.0, 0.0, z)), 'ground') is None
    assert ipm_read_formula(30.0, z) == np.inf
print('\n✅ 公式与数值一致，且 z >= H 时两者都给「无解」')

## 4 · 最反直觉的一条推论：相对误差与距离无关

把公式整理一下：

$$\frac{d_{\text{read}}-d}{d}=\frac{z}{H-z}$$

**右边完全不含 $d$。** 所以这个误差不是「远处才严重」——
它在每个距离上都是同一个倍数。
这直接否掉了一个很常见的上线计划：「先做近处，远处以后再修」。

In [ ]:
print(f"{'z (m)':>6s} {'相对误差':>10s}   " + ''.join(f'{d}m 读出'.rjust(12) for d in [10,30,50,100]))
for z in [0.0, 0.3, 0.5, 1.0, 1.2, 1.4]:
    rel = z / (H - z)
    reads = [ipm_read_formula(d, z) for d in [10, 30, 50, 100]]
    print(f'{z:6.1f} {rel*100:9.1f}%   ' + ''.join(f'{r:12.2f}' for r in reads))

# 断言：同一个 z 下，各距离的**倍数**完全相同
for z in [0.3, 0.8, 1.0, 1.4]:
    ratios = [ipm_read_formula(d, z) / d for d in [8, 10, 30, 50, 100, 200]]
    assert max(ratios) - min(ratios) < 1e-12, '倍数必须与距离无关'
    assert abs(ratios[0] - H / (H - z)) < 1e-12
print('\n✅ 倍数 = H/(H−z)，与距离无关（各距离间差异 < 1e-12）')

# 横向也被同一个倍数缩放
P = (30.0, 2.0, 1.0)
bp = backproject(project(P), 'ground')
k = H / (H - P[2])
assert abs(bp[0] - P[0]*k) < 1e-9 and abs(bp[1] - P[1]*k) < 1e-9
print(f'✅ 横向同样被放大 {k:.1f} 倍：真 Y={P[1]} -> 读出 Y={bp[1]:.2f}'
      '  → **整个 BEV 图被径向拉伸，而不是平移**')

## 5 · 把「1° 外参误差」换算成「多少个像素」

标定误差和检测误差最终都变成米。放到同一把尺子上比，才知道该投入哪一边。

In [ ]:
print(f"{'距离':>6s} {'1px 定位误差':>14s} {'1° 俯仰误差':>14s} {'1° 相当于':>12s}")
budget = []
for d in [10, 20, 30, 50, 80]:
    uv = project((d, 0.0, 0.0))
    d0 = backproject(uv, 'ground')[0]
    e_px = abs(backproject(uv + np.array([0.0, 1.0]), 'ground')[0] - d0)
    cands = []
    for sign in (+1.0, -1.0):
        bp = backproject(uv, 'ground', pitch=sign * 1.0)
        cands.append(abs((np.inf if bp is None else bp[0]) - d0))
    e_deg = max(cands)
    budget.append((d, e_px, e_deg))
    print(f'{d:5d}m {e_px:13.3f}m {e_deg:13.2f}m {e_deg/e_px:11.1f} px')

# 两个结论都要成立
assert all(e_deg / e_px > 20 for _, e_px, e_deg in budget), '1° 应远大于 1px'
ratios = [e_deg / e_px for _, e_px, e_deg in budget]
assert ratios == sorted(ratios), '比值应随距离单调上升'
print('\n✅ 1° 外参误差相当于 24–52 个像素的定位误差，且比值随距离上升')
print('   → **标定的收益远大于同等工程量投在检测器定位精度上**')

# 误差是不对称的：抬头和低头不一样
uv = project((50.0, 0.0, 0.0)); d0 = backproject(uv, 'ground')[0]
up   = backproject(uv, 'ground', pitch=-1.0)[0]
down = backproject(uv, 'ground', pitch=+1.0)[0]
print(f'\n50m 处：pitch+1°(低头) -> {down:7.2f}m   pitch-1°(抬头) -> {up:7.2f}m')
print(f'   两侧偏差 {abs(down-d0):.2f}m vs {abs(up-d0):.2f}m —— **高度不对称**')
lin = H / np.sin(np.arctan2(H, 50.0))**2 * np.deg2rad(1)
print(f'   一阶近似给 ±{lin:.2f}m —— **两侧都不对**（远处线性化失效）')
assert abs(up - d0) > 3 * abs(down - d0), '抬头一侧的误差应远大于低头一侧'

## 6 · 小结：本模块建立的东西

| 结论 | 数值 | 在哪个模块被展开 |
|---|---|---|
| 正向投影唯一、反向缺一个自由度 | 补法只有四种 | 全课 |
| 地面假设被破坏 | $d_{\text{read}}=d\cdot H/(H-z)$，相对偏差 < 1e-12 | 模块 04 |
| 相对误差与距离无关 | $z/(H-z)$，各距离差异 < 1e-12 | 模块 04 |
| $z\ge H$ 时 IPM 无解 | 而限速牌就在 2.0–2.5 m | 模块 04 |
| 1° 外参 = 24–52 px | 且比值随距离上升 | 模块 02 |
| 俯仰误差高度不对称 | 50m 处 −18m / +70m | 模块 02 |

## ✏️ 练习 1：批量投影

实现 `project_batch(P, pitch=0.0)`：输入形状 `(N,3)` 的车体坐标数组，
返回 `(N,2)` 的像素数组；**相机后方的点对应行填 `np.nan`**。

要求：不许在 Python 里逐点循环（用矩阵运算），且与逐点 `project()` 完全一致。

In [ ]:
def project_batch(P, pitch=0.0):
    """(N,3) 车体坐标 -> (N,2) 像素；相机后方的点填 nan。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
PTS = np.array([list(v) for v in SCENE.values()] + [
    [-5.0, 0.0, 0.0],      # 车后
    [0.0, 0.0, 0.0],       # 正下方
    [100.0, -8.0, 3.0],
])

got = project_batch(PTS)
assert got.shape == (len(PTS), 2), got.shape

for i, P in enumerate(PTS):
    ref = project(P)
    if ref is None:
        assert np.all(np.isnan(got[i])), f'第 {i} 点在相机后方，应为 nan'
    else:
        assert np.allclose(got[i], ref, atol=1e-9), (i, got[i], ref)

got_p = project_batch(PTS, pitch=2.5)
for i, P in enumerate(PTS):
    ref = project(P, pitch=2.5)
    if ref is not None:
        assert np.allclose(got_p[i], ref, atol=1e-9), (i, got_p[i], ref)

n_nan = int(np.isnan(got[:, 0]).sum())
print(f'{len(PTS)} 个点：{len(PTS)-n_nan} 个成像、{n_nan} 个在相机后方')
print('✅ 练习 1 通过：批量与逐点完全一致，且后方点被标成 nan')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def project_batch(P, pitch=0.0):
    P = np.atleast_2d(np.asarray(P, float))
    P_c = (P - CAM_T) @ R_vc(pitch)            # (N,3) @ (3,3) == (R.T @ v).T
    out = np.full((len(P), 2), np.nan)
    ok = P_c[:, 2] > 1e-9
    out[ok, 0] = CX + F * P_c[ok, 0] / P_c[ok, 2]
    out[ok, 1] = CY + F * P_c[ok, 1] / P_c[ok, 2]
    return out

for i, P in enumerate(PTS):
    ref = project(P)
    g = project_batch(PTS)[i]
    assert (np.all(np.isnan(g)) if ref is None else np.allclose(g, ref, atol=1e-9))
print('✅ 参考答案 1 通过（注意 (P-t) @ R 等价于 R.T @ (P-t) 的批量写法）')

## ✏️ 练习 2：已知尺寸测距

法规把交通标志的物理尺寸钉死了（例如圆形限速牌直径 0.6 / 0.8 / 1.2 m）。
于是「已知尺寸」是 TSR 里最实用的一种补法。

实现 `dist_from_size(size_px, size_m)`：由像素边长与真实边长求**沿光轴的深度**，
并实现 `dist_from_size_err(size_px, size_m, px_err)`：
给定像素测量误差 `px_err`，返回 `(深度下界, 深度上界)`。

> 自测里会顺带验证一条定律：**不确定度 $\propto d^2$** ——
> 这与模块 03 的三角测量 $Z^2/(Bf)$ 是同一条，只是把基线 $B$ 换成了物体尺寸 $S$。

In [ ]:
def dist_from_size(size_px, size_m):
    """相似三角形测距：返回沿光轴深度 (m)。"""
    # TODO
    raise NotImplementedError

def dist_from_size_err(size_px, size_m, px_err):
    """返回 (下界, 上界)：像素边长在 ±px_err 内变动时深度的范围。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
SIGN_M = 0.8                      # 直径 0.8m 的限速牌
TRUE_D = [10.0, 30.0, 50.0, 80.0]
sizes  = [F * SIGN_M / d for d in TRUE_D]

for d, s in zip(TRUE_D, sizes):
    assert abs(dist_from_size(s, SIGN_M) - d) < 1e-9, (d, s)

lo, hi = dist_from_size_err(sizes[1], SIGN_M, 1.0)
assert lo < 30.0 < hi
assert abs(hi - lo) > abs(dist_from_size_err(sizes[0], SIGN_M, 1.0)[1]
                          - dist_from_size_err(sizes[0], SIGN_M, 1.0)[0]), \
    '远处同样的像素误差应造成更大的深度不确定度'

print(f"{'真距':>6s} {'像素边长':>10s} {'±1px 的深度区间':>24s} {'相对宽度':>9s} {'宽度/d²':>12s}")
rel_w, norm_w = [], []
for d, s in zip(TRUE_D, sizes):
    lo, hi = dist_from_size_err(s, SIGN_M, 1.0)
    rel_w.append((hi - lo) / d)
    norm_w.append((hi - lo) / d ** 2)
    print(f'{d:5.0f}m {s:9.2f}px   [{lo:8.2f}, {hi:8.2f}] {100*(hi-lo)/d:8.1f}% {(hi-lo)/d**2:12.4e}')

assert rel_w == sorted(rel_w), '相对宽度应随距离单调上升'
# 关键定律：不确定度 ∝ d²   （宽度 ≈ 2·e_px·d²/(f·S)）
# 所以「宽度/d²」应当近乎常数——这与模块 03 三角测量的 Z²/(B·f) 是**同一条定律**，
# 只是把「基线 B」换成了「物体的物理尺寸 S」
assert (max(norm_w) - min(norm_w)) / min(norm_w) < 0.01, '宽度/d² 应在 1% 内恒定'
print(f'\n宽度/d² 的波动只有 {100*(max(norm_w)-min(norm_w))/min(norm_w):.2f}%'
      '  → **不确定度 ∝ d²**')
print('✅ 练习 2 通过：已知尺寸能给出距离**和**不确定度，且服从 d² 定律')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def dist_from_size(size_px, size_m):
    return F * float(size_m) / float(size_px)

def dist_from_size_err(size_px, size_m, px_err):
    s_hi = size_px + px_err        # 看起来更大 -> 更近
    s_lo = max(size_px - px_err, 1e-9)
    return (dist_from_size(s_hi, size_m), dist_from_size(s_lo, size_m))

for d, s in zip(TRUE_D, sizes):
    assert abs(dist_from_size(s, SIGN_M) - d) < 1e-9
lo, hi = dist_from_size_err(sizes[3], SIGN_M, 1.0)
print(f'80m 处 ±1px -> [{lo:.2f}, {hi:.2f}] m，宽度 {hi-lo:.2f} m')
print('✅ 参考答案 2 通过（注意区间不对称：深度是像素边长的倒数关系）')

## ✏️ 练习 3：把中心公式的三条推论写成检查

实现 `ipm_audit(h)`，返回一个 dict，逐条验证：

- `'rel_bias_at'` —— `{z: 相对误差}`，对 `z in (0.0, 0.5, 1.0)`
- `'dist_free'` —— bool：相对误差是否与距离无关（在 `d in (10,50,200)` 上验证）
- `'diverges_at'` —— float：发散点（读出趋于无穷的 z）
- `'no_solution_above'` —— float：从哪个 z 起无解

这四项就是模块 04 的验收清单。

In [ ]:
def ipm_audit(h):
    """返回 dict(rel_bias_at, dist_free, diverges_at, no_solution_above)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
for h in [1.2, 1.5, 2.4]:
    a = ipm_audit(h)
    assert set(a) == {'rel_bias_at', 'dist_free', 'diverges_at', 'no_solution_above'}
    assert a['dist_free'] is True, '相对误差必须与距离无关'
    assert abs(a['diverges_at'] - h) < 1e-12
    assert abs(a['no_solution_above'] - h) < 1e-12
    assert abs(a['rel_bias_at'][0.0]) < 1e-12
    assert abs(a['rel_bias_at'][1.0] - 1.0 / (h - 1.0)) < 1e-12

a = ipm_audit(1.5)
print('h=1.5m 的审计结果：')
for z, r in a['rel_bias_at'].items():
    print(f'  离地 {z:.1f}m -> 相对误差 {r*100:8.1f}%')
print(f"  与距离无关: {a['dist_free']}   发散于 z={a['diverges_at']}m"
      f"   z>={a['no_solution_above']}m 无解")
print('\n✅ 练习 3 通过：三条推论都成立，且对任意相机高度都成立')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def ipm_audit(h):
    rel = {}
    for z in (0.0, 0.5, 1.0):
        rel[z] = z / (h - z)
    free = True
    for z in (0.0, 0.5, 1.0):
        rs = [ipm_read_formula(d, z, h) / d for d in (10.0, 50.0, 200.0)]
        if max(rs) - min(rs) > 1e-12:
            free = False
    return {'rel_bias_at': rel, 'dist_free': free,
            'diverges_at': float(h), 'no_solution_above': float(h)}

for h in [1.2, 1.5, 2.4]:
    a = ipm_audit(h)
    assert a['dist_free'] and abs(a['diverges_at'] - h) < 1e-12
print('✅ 参考答案 3 通过')

## ✏️ 练习 4：误差预算表

实现 `error_budget(d, *, px=1.0, pitch_deg=1.0, dt_ms=33.0, speed_kmh=120.0)`，
返回一个 dict，把四个来源都换算成**米**，并给出 `'dominant'`（最大的那一项的名字）：

- `'pixel'` —— `px` 个像素的定位误差
- `'pitch'` —— `pitch_deg` 度俯仰误差（取抬头/低头两侧的**最大**值）
- `'latency'` —— `dt_ms` 毫秒时间戳误差 × 车速
- `'assumption'` —— 把一块离地 1.0 m 的标志当地面点处理

这张表是本课的核心交付物：**它让「该修哪一个」变成一个可计算的问题。**

In [ ]:
def error_budget(d, *, px=1.0, pitch_deg=1.0, dt_ms=33.0, speed_kmh=120.0):
    """把四个误差源都换算成米，并指出主导项。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
b30 = error_budget(30.0)
assert set(b30) == {'pixel', 'pitch', 'latency', 'assumption', 'dominant'}
assert b30['dominant'] in ('pixel', 'pitch', 'latency', 'assumption')

# 时间戳一项与距离无关，纯运动学
assert abs(error_budget(10.0)['latency'] - error_budget(80.0)['latency']) < 1e-12
assert abs(b30['latency'] - 120/3.6*0.033) < 1e-9

# 像素与俯仰两项都随距离增长
for k in ('pixel', 'pitch'):
    vals = [error_budget(d)[k] for d in (10, 20, 30, 50, 80)]
    assert vals == sorted(vals), f'{k} 应随距离单调上升'

print(f"{'距离':>6s} {'像素':>9s} {'俯仰1°':>10s} {'时延33ms':>10s} {'假设破坏':>10s}  主导项")
for d in [10, 20, 30, 50, 80]:
    b = error_budget(d)
    print(f"{d:5d}m {b['pixel']:8.3f}m {b['pitch']:9.2f}m "
          f"{b['latency']:9.3f}m {b['assumption']:9.2f}m  {b['dominant']}")

assert error_budget(10.0)['dominant'] == 'assumption', '近处：假设破坏最致命'
assert error_budget(80.0)['dominant'] == 'pitch', '远处：标定误差反超'
print('\n✅ 练习 4 通过：主导项随距离切换——'
      '近处「假设用错」最致命，远处「标定」反超')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def error_budget(d, *, px=1.0, pitch_deg=1.0, dt_ms=33.0, speed_kmh=120.0):
    uv = project((d, 0.0, 0.0))
    d0 = backproject(uv, 'ground')[0]

    e_px = abs(backproject(uv + np.array([0.0, px]), 'ground')[0] - d0)

    e_pitch = 0.0
    for sign in (+1.0, -1.0):
        bp = backproject(uv, 'ground', pitch=sign * pitch_deg)
        v = np.inf if bp is None else bp[0]
        e_pitch = max(e_pitch, abs(v - d0))

    e_lat = speed_kmh / 3.6 * (dt_ms / 1000.0)
    e_ass = abs(ipm_read_formula(d, 1.0) - d)

    out = {'pixel': e_px, 'pitch': e_pitch, 'latency': e_lat, 'assumption': e_ass}
    out['dominant'] = max(out, key=out.get)
    return out

assert error_budget(10.0)['dominant'] == 'assumption'
assert error_budget(80.0)['dominant'] == 'pitch'
print('✅ 参考答案 4 通过')

## 🧪 真实工程胶囊：把这一层接到真实栈上

本课自己实现投影是为了看清误差从哪来。真实项目里对应的东西：

```python
# ── 1) OpenCV：投影与去畸变（模块 01 会自己实现一遍）──
import cv2
uv, _ = cv2.projectPoints(P_obj, rvec, tvec, K, dist)      # 正向
xy_n  = cv2.undistortPoints(uv, K, dist)                    # 去畸变到归一化平面

# ── 2) ROS TF2：外参不该写在代码里，应该来自变换树 ──
#   坐标系命名遵循 REP-105：base_link / camera_link / map
tf = tf_buffer.lookup_transform('base_link', 'camera_front',
                                stamp, timeout=rospy.Duration(0.05))
#   ↑ 关键是 stamp：**用哪一时刻的外参**，而不是「当前」的
#     悬挂动态让 pitch 在行驶中变化，模块 02 第 6 节量它的幅度

# ── 3) 把「假设」写进接口，而不是写在注释里 ──
@dataclass(frozen=True)
class Detection3D:
    uv: tuple
    range_m: float
    assumption: Literal['ground', 'known_height', 'known_size', 'stereo']
    sigma_m: float          # ← 与 range 一起给，否则下游无法融合
#   下游（规控）必须能读到「这个米数是怎么来的」
#   而 sigma 的算法就是练习 4 的误差预算表

# ── 4) 标定件的版本化（模块 02 第 7 节）──
#   intrinsics.yaml / extrinsics.yaml 必须带：
#     标定日期 · 重投影误差的中位数与 P95 · 标定板规格 · 采集张数
#   **没有重投影误差的标定文件不允许上车**
```

> **一条可以立刻用的检查**：把你现有栈里所有「像素 → 米」的调用点找出来，
> 逐个回答「这里用的是哪一种补法」。
> 本课的经验是：**总有几处答不上来，而它们通常就是「距离总是偏大」的来源。**